# Agentic Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local agentic pipeline for English-to-Chinese cross-lingual dialogue summarization.

The pipeline uses three local small language model agents. Agent 1 first generates an English summary from the original dialogue. Agent 2 translates the English summary into Chinese. Agent 3 then evaluates and revises the Chinese summary for faithfulness, coverage, and fluency.

```text
English Dialogue
→ Agent 1: English Summarization Agent
→ Agent 2: Chinese Translation Agent
→ Agent 3: Evaluation-Revision Agent
→ Final Chinese Summary
```

The pipeline consists of three agents:

```text
Agent 1: English Summarization Agent
Input: original English dialogue
Output: initial English summary

Agent 2: Chinese Translation Agent
Input: initial English summary
Output: initial Chinese summary

Agent 3: Evaluation-Revision Agent
Input: original dialogue + English summary + initial Chinese summary
Output: final revised Chinese summary
```

The local small language models are served through Ollama. The notebook controls the agent workflow, prompt design, input/output processing, and result saving.

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the models locally.

### Recommended model setup

```bash
ollama pull qwen3.5:9b
ollama pull translategemma:12b
```

If `translategemma:12b` is too slow on your machine, use:

```bash
ollama pull translategemma:4b
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages
# Run this only once if the packages are not installed.

!pip install requests pandas tqdm



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [3]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# Agent models
SEMANTIC_UNDERSTANDING_MODEL = "qwen3.5:9b"
SUMMARY_GENERATION_MODEL = "qwen3.5:9b"
TRANSLATION_MODEL = "translategemma:12b"

# If translategemma:12b is too slow, use:
# TRANSLATION_MODEL = "translategemma:4b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

# Input dataset path
RAW_TEST_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files
FULL_OUTPUT_PATH = OUTPUT_DIR / "semantic_summary_translation_outputs.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "semantic_summary_translation_final_summaries.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "semantic_summary_translation_errors.jsonl"

print("Raw test path:", RAW_TEST_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Raw test path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_summary_translation_outputs.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_summary_translation_final_summaries.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_summary_translation_errors.jsonl


In [4]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['translategemma:12b', 'qwen3.5:9b']


In [5]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [6]:
# Cell 5: Prompt templates

SEMANTIC_UNDERSTANDING_PROMPT = """You are a semantic understanding agent for English dialogues.
Your task is to extract the core meaning of the dialogue into a structured JSON representation.

Guidelines:
1. Use only information explicitly supported by the dialogue.
2. Focus on participants, key events, requests/actions, important entities, and uncertainties.
3. Keep the representation concise but complete enough for downstream summarization.
4. For each key event or request/action, include short evidence from the dialogue.
5. Output ONLY valid JSON. Do not include markdown, comments, or explanations.

Output schema:
{{
  "participants": ["string"],
  "key_events": [
    {{
      "actor": "string",
      "action": "string",
      "target_or_object": "string",
      "evidence": "short quote or paraphrase from the dialogue"
    }}
  ],
  "requests_or_actions": [
    {{
      "person": "string",
      "action": "string",
      "evidence": "short quote or paraphrase from the dialogue"
    }}
  ],
  "important_entities": ["string"],
  "uncertainties": ["string"]
}}

Input dialogue:
{dialogue}

JSON output:
"""


SUMMARY_GENERATION_PROMPT = """You are an expert English dialogue summarization agent.
Your task is to generate a concise English summary using both the original dialogue and the structured semantic representation.

Guidelines:
1. Capture the main intent, key events, important entities, and final outcome.
2. Preserve factual details from the dialogue.
3. Do not add unsupported information.
4. Prefer one or two concise sentences.
5. Output ONLY the English summary. Do not include explanations.

Original dialogue:
{dialogue}

Structured semantic representation:
{semantic_representation}

English summary:
"""


TRANSLATION_PROMPT = """You are an expert English-to-Simplified-Chinese translation agent.
Your task is to translate the English summary into a natural and faithful Simplified Chinese summary.

Guidelines:
1. Preserve the meaning of the English summary.
2. Preserve names, entities, actions, and temporal information.
3. Use the structured semantic representation only as context to avoid mistranslation.
4. Do not add information that is not present in the English summary or semantic representation.
5. Output ONLY the final Simplified Chinese summary. Do not include explanations.

Structured semantic representation:
{semantic_representation}

English summary:
{english_summary}

Chinese summary:
"""

In [7]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [8]:
# Cell 7: Agent functions

def extract_json_object(text: str) -> Dict[str, Any]:
    """Extract and parse a JSON object from a model response.

    Small language models sometimes wrap JSON in markdown fences or add extra text.
    This helper tries to recover the JSON object and returns a safe fallback if parsing fails.
    """
    raw = text.strip()

    # Remove markdown code fences if the model returns ```json ... ```.
    if raw.startswith("```"):
        lines = raw.splitlines()
        raw = "\n".join(
            line for line in lines
            if not line.strip().startswith("```")
        ).strip()

    # First try direct JSON parsing.
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            return parsed
    except json.JSONDecodeError:
        pass

    # Then try extracting the substring between the first "{" and the last "}".
    start = raw.find("{")
    end = raw.rfind("}")

    if start != -1 and end != -1 and end > start:
        candidate = raw[start:end + 1]
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict):
                return parsed
        except json.JSONDecodeError:
            pass

    # Safe fallback: keep the raw output for debugging.
    return {
        "participants": [],
        "key_events": [],
        "requests_or_actions": [],
        "important_entities": [],
        "uncertainties": ["Failed to parse semantic JSON output."],
        "raw_output": text,
    }


def format_semantic_representation(semantic_representation: Dict[str, Any]) -> str:
    """Format the semantic representation as readable JSON text."""
    return json.dumps(
        semantic_representation,
        ensure_ascii=False,
        indent=2,
    )


def semantic_understanding_agent(dialogue: str) -> Dict[str, Any]:
    """Agent 1: Extract a structured semantic representation from the dialogue."""
    prompt = SEMANTIC_UNDERSTANDING_PROMPT.format(dialogue=dialogue)

    response = call_ollama(
        model=SEMANTIC_UNDERSTANDING_MODEL,
        prompt=prompt,
        temperature=0.1,
    )

    return extract_json_object(response)


def summary_generation_agent(
    dialogue: str,
    semantic_representation: Dict[str, Any],
) -> str:
    """Agent 2: Generate an English summary from the dialogue and semantic representation."""
    prompt = SUMMARY_GENERATION_PROMPT.format(
        dialogue=dialogue,
        semantic_representation=format_semantic_representation(semantic_representation),
    )

    return call_ollama(
        model=SUMMARY_GENERATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )


def translation_agent(
    english_summary: str,
    semantic_representation: Dict[str, Any],
) -> str:
    """Agent 3: Translate the English summary into Simplified Chinese."""
    prompt = TRANSLATION_PROMPT.format(
        english_summary=english_summary,
        semantic_representation=format_semantic_representation(semantic_representation),
    )

    return call_ollama(
        model=TRANSLATION_MODEL,
        prompt=prompt,
        temperature=0.1,
    )

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [9]:
# Cell 8: Three-agent pipeline

def run_three_agent_pipeline(example: Dict[str, Any]) -> Dict[str, Any]:
    """Run the semantic understanding → summary generation → translation pipeline on one example."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    # Agent 1: Semantic understanding
    semantic_representation = semantic_understanding_agent(dialogue)

    # Agent 2: English summary generation
    english_summary = summary_generation_agent(
        dialogue=dialogue,
        semantic_representation=semantic_representation,
    )

    # Agent 3: Chinese translation
    final_chinese_summary = translation_agent(
        english_summary=english_summary,
        semantic_representation=semantic_representation,
    )

    return {
        "id": sample_id,
        "dialogue": dialogue,
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,
        "agent1_semantic_representation": semantic_representation,
        "agent1_semantic_representation_json": format_semantic_representation(semantic_representation),
        "agent2_english_summary": english_summary,
        "agent3_chinese_summary": final_chinese_summary,
        "final_summary": final_chinese_summary,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect the intermediate outputs between agents.


In [10]:
# Cell 9: Load top examples from the original test.json file

def load_top_examples_from_test_json(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the top n examples from the original JSON test file."""
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"test_{i+1:05d}",
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_top_examples_from_test_json(RAW_TEST_PATH, n=5)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 5 examples.
First example:
{'id': 'test_00001', 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye", 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.", 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。'}


In [11]:
# Cell 10: Run the three-agent pipeline for the first example from test.json

result = run_three_agent_pipeline(test_data[0])
result

{'id': 'test_00001',
 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye",
 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.",
 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。',
 'agent1_semantic_representation': {'participants': ['Hannah', 'Amanda'],
  'key_events': [{'actor': 'Hannah',
    'action': "requests Betty's phone number",
    'target_or_object': "Betty's phone number",
    'evidence': "Hey, do you have Betty's number?"},
   {'actor': 'Amanda',
    'action': "searches for Betty's number",
    'target_or_object': "Betty's phon

In [12]:
# Cell 11: Print three-agent pipeline result clearly

def print_three_agent_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1: Semantic Representation ===")
    print(result["agent1_semantic_representation_json"])
    print()

    print("=== Agent 2: English Summary ===")
    print(result["agent2_english_summary"])
    print()

    print("=== Agent 3: Chinese Summary ===")
    print(result["agent3_chinese_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Final Chinese Summary ===")
    print(result["final_summary"])


print_three_agent_result(result)

=== Original Dialogue ===
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

=== Agent 1: Semantic Representation ===
{
  "participants": [
    "Hannah",
    "Amanda"
  ],
  "key_events": [
    {
      "actor": "Hannah",
      "action": "requests Betty's phone number",
      "target_or_object": "Betty's phone number",
      "evidence": "Hey, do you have Betty's number?"
    },
    {
      "actor": "Amanda",
      "action": "searches for Betty's number",
      "target_or_object": "Betty's phone number",
      "evidence": "Lemme check"
    },
    {
      "actor": "Amanda",
      "action": "suggests contacting Larry",
      "target_or_ob

## 4. Save Results

This saves all intermediate outputs and the final output.


In [13]:
# Cell 12: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_summary_translation_outputs.jsonl
CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_summary_translation_final_summaries.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_summary_translation_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed result to `results/agentic_outputs.jsonl`.

If the notebook stops, already processed examples remain saved.


In [14]:
# Cell 13: Batch inference with three-agent pipeline

MAX_EXAMPLES = 5
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running semantic-summary-translation pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        continue

    try:
        record = run_three_agent_pipeline(ex)
        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running semantic-summary-translation pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Finished. Outputs saved to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_summary_translation_outputs.jsonl


## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.


In [15]:
# Cell 14: Export final summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "dialogue": record.get("dialogue", ""),
        "final_summary": record.get("final_summary", ""),
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),
        "agent1_semantic_representation_json": record.get("agent1_semantic_representation_json", ""),
        "agent2_english_summary": record.get("agent2_english_summary", ""),
        "agent3_chinese_summary": record.get("agent3_chinese_summary", ""),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/semantic_summary_translation_final_summaries.csv


,id,dialogue,final_summary,reference_english_summary,reference_chinese_summary,agent1_semantic_representation_json,agent2_english_summary,agent3_chinese_summary
0,test_00001,"Hannah: Hey, do you have Betty's number?\nAman...",汉娜向曼达询问贝蒂的电话号码，但曼达没有找到。曼达建议联系拉里，汉娜对此感到犹豫。最终，汉娜...,Hannah needs Betty's number but Amanda doesn't...,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。,"{\n ""participants"": [\n ""Hannah"",\n ""Am...","Hannah asks Amanda for Betty's phone number, b...",汉娜向曼达询问贝蒂的电话号码，但曼达没有找到。曼达建议联系拉里，汉娜对此感到犹豫。最终，汉娜...
1,test_00002,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric:...,Eric和Rob讨论了脱口秀节目“MACHINE”。Eric询问是否只有这一期节目，Rob在...,Eric and Rob are going to watch a stand-up on ...,埃里克和罗伯要在youtube上看一场单口相声。,"{\n ""participants"": [\n ""Eric"",\n ""Rob""...","Eric and Rob discuss the stand-up show ""MACHIN...",Eric和Rob讨论了脱口秀节目“MACHINE”。Eric询问是否只有这一期节目，Rob在...
2,test_00003,"Lenny: Babe, can you help me with something?\r...",Lenny 询问 Bob 帮助他选择三条裤子。Bob 推荐了第一条裤子，并解释说购买更多裤子...,Lenny can't decide which trousers to buy. Bob ...,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。,"{\n ""participants"": [\n ""Lenny"",\n ""Bob...",Lenny asks Bob for help choosing between three...,Lenny 询问 Bob 帮助他选择三条裤子。Bob 推荐了第一条裤子，并解释说购买更多裤子...
3,test_00004,"Will: hey babe, what do you want for dinner to...",威廉询问艾玛晚餐的安排以及她回家的时间。艾玛表示今晚不想做饭，也没有食欲，并确认她很快会回家...,Emma will be home soon and she will let Will k...,艾玛很快就会回家，而且她会告诉威尔。,"{\n ""participants"": [\n ""Will"",\n ""Emma...",Will asks Emma about dinner and her return tim...,威廉询问艾玛晚餐的安排以及她回家的时间。艾玛表示今晚不想做饭，也没有食欲，并确认她很快会回家...
4,test_00005,"Ollie: Hi , are you in Warsaw\r\nJane: yes, ju...",简 (Jane) 刚刚回到华沙，她与奥利 (Ollie) 讨论了派对、潜在的午餐以及前往摩洛...,Jane is in Warsaw. Ollie and Jane has a party....,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...,"{\n ""participants"": [],\n ""key_events"": [],\...","Jane, who has just returned to Warsaw, coordin...",简 (Jane) 刚刚回到华沙，她与奥利 (Ollie) 讨论了派对、潜在的午餐以及前往摩洛...


In [16]:
# Cell 15: Compare generated Chinese summary with the reference Chinese summary

comparison_columns = [
    "id",
    "agent2_english_summary",
    "agent3_chinese_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,agent2_english_summary,agent3_chinese_summary,reference_chinese_summary
0,test_00001,"Hannah asks Amanda for Betty's phone number, b...",汉娜向曼达询问贝蒂的电话号码，但曼达没有找到。曼达建议联系拉里，汉娜对此感到犹豫。最终，汉娜...,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。
1,test_00002,"Eric and Rob discuss the stand-up show ""MACHIN...",Eric和Rob讨论了脱口秀节目“MACHINE”。Eric询问是否只有这一期节目，Rob在...,埃里克和罗伯要在youtube上看一场单口相声。
2,test_00003,Lenny asks Bob for help choosing between three...,Lenny 询问 Bob 帮助他选择三条裤子。Bob 推荐了第一条裤子，并解释说购买更多裤子...,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。
3,test_00004,Will asks Emma about dinner and her return tim...,威廉询问艾玛晚餐的安排以及她回家的时间。艾玛表示今晚不想做饭，也没有食欲，并确认她很快会回家...,艾玛很快就会回家，而且她会告诉威尔。
4,test_00005,"Jane, who has just returned to Warsaw, coordin...",简 (Jane) 刚刚回到华沙，她与奥利 (Ollie) 讨论了派对、潜在的午餐以及前往摩洛...,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...


In [17]:
# Cell 16: Inspect the semantic representation and final output

if not df.empty:
    inspection_columns = [
        "id",
        "agent1_semantic_representation_json",
        "agent2_english_summary",
        "agent3_chinese_summary",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")

,id,agent1_semantic_representation_json,agent2_english_summary,agent3_chinese_summary
0,test_00001,"{\n ""participants"": [\n ""Hannah"",\n ""Am...","Hannah asks Amanda for Betty's phone number, b...",汉娜向曼达询问贝蒂的电话号码，但曼达没有找到。曼达建议联系拉里，汉娜对此感到犹豫。最终，汉娜...
1,test_00002,"{\n ""participants"": [\n ""Eric"",\n ""Rob""...","Eric and Rob discuss the stand-up show ""MACHIN...",Eric和Rob讨论了脱口秀节目“MACHINE”。Eric询问是否只有这一期节目，Rob在...
2,test_00003,"{\n ""participants"": [\n ""Lenny"",\n ""Bob...",Lenny asks Bob for help choosing between three...,Lenny 询问 Bob 帮助他选择三条裤子。Bob 推荐了第一条裤子，并解释说购买更多裤子...
3,test_00004,"{\n ""participants"": [\n ""Will"",\n ""Emma...",Will asks Emma about dinner and her return tim...,威廉询问艾玛晚餐的安排以及她回家的时间。艾玛表示今晚不想做饭，也没有食欲，并确认她很快会回家...
4,test_00005,"{\n ""participants"": [],\n ""key_events"": [],\...","Jane, who has just returned to Warsaw, coordin...",简 (Jane) 刚刚回到华沙，她与奥利 (Ollie) 讨论了派对、潜在的午餐以及前往摩洛...
